## Web Scraping - War in Ukraine News Analyzer

This notebook is inspired by and builds upon the work in the GitHub repository [NewsAutoAnalysisUKR](https://github.com/Op27/NewsAutoAnalysisUKR).  

---

### Overview

In this notebook, we will web scrape and analyze latest news articles related to the war in Ukraine, sourced from the [BBC News website](https://www.bbc.com/news/war-in-ukraine). It uses various tools and techniques in data science and natural language processing (NLP) to scrape, process, and visualize key insights from the articles.


### Key Features
1. **Web Scraping**: The notebook uses `BeautifulSoup` to extract information (headlines, URLs, and full-text) from the BBC News website, focusing on the war in Ukraine.
2. **Text Analysis**: Processes the article content to identify the most frequently mentioned words, removing common "stop words" to enhance the analysis.
3. **Visualization**:
   - **Word Cloud**: Highlights the most prominent keywords from news articles.
   - **Bar Chart**: Displays the frequency of the top words after filtering out common stop words.
4. **GPT Summarization**: Leverages OpenAI's GPT models to summarize news articles, capturing the essence of the news in a professional format.
5. **Report Generation**: Automatically compiles all data and visuals into a clean and structured Word document.

### Steps in the Notebook
1. **Scrape and Save Articles**: Scrape news articles from the BBC website and save the content into a text file for further processing.
2. **Generate Visualizations**: Process the text to create insightful visualizations like word clouds and bar charts.
3. **Summarize with GPT**: Use GPT models to summarize the scraped articles, focusing on recurring themes, humanitarian issues, and key mentions.
4. **Compile a Report**: Integrate the scraped articles, visualizations, and summaries into a Word document for easy sharing and analysis.


In [ ]:
## Install required libraries
#!pip install python-docx beautifulsoup4 wordcloud openai matplotlib requests

In [ ]:
# Import required libraries
import requests
from bs4 import BeautifulSoup
import re
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
import docx
import datetime
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
import os
from docx.shared import Inches
from openai import OpenAI

### Step 1: Scrape information from the web and save them to a text file

In [ ]:
# Add headers to avoid consent/lean pages
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.bbc.com/"
}

def scrape_and_save_short():
    URL = 'https://www.bbc.com/news/war-in-ukraine'
    response = requests.get(URL, headers=HEADERS, timeout=30)  # FIX
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    articles = soup.find_all('div', class_='sc-9d830f2a-0 cKjkGu')
    headlines_urls_and_descriptions = []

    for article in articles:
        headline = article.find('h2') or article.find('h3')  # small safety
        description_tag = article.find('p', {'data-testid': 'card-description'})
        # FIX: get the link nearest to the headline
        link = None
        if headline:
            link = headline.find_parent('a')  # headline wrapped by anchor?
            if not link:
                # try any anchor inside the same card
                link = article.find('a', href=True)
            if not link:
                # last resort: the next anchor after the headline
                link = headline.find_next('a', href=True)

        if headline and link and link.get('href'):
            headline_text = headline.get_text(strip=True)
            description_text = description_tag.get_text(strip=True) if description_tag else ""
            href = link['href']
            url = 'https://www.bbc.com' + href if href.startswith('/') else href
            headlines_urls_and_descriptions.append((headline_text, url, description_text))

    with open('articles.txt', 'w', encoding='utf-8') as file:
        for headline, url, description in headlines_urls_and_descriptions:
            file.write(f"{headline} (BBC): {description} URL:{url}\n\n")

#### Verify what the parser captures

In [ ]:
# Verify what the parser captures
response = requests.get('https://www.bbc.com/news/war-in-ukraine', headers=HEADERS, timeout=30)  # FIX
soup = BeautifulSoup(response.content, 'html.parser')
articles = soup.find_all('div', class_='sc-9d830f2a-0 cKjkGu')

# Print the captured content
for i, article in enumerate(articles):
  print(f"Article {i+1}:\n")
  print(article.get_text(strip=True))  # Prints the text within each <div>
  print("-" * 50)
  if i > 1: break

The parser only captures the headlines. Let's modify the code further so it scrapes the entire pages as well.

In [ ]:
def scrape_and_save():
    # Step 1: Scrape headlines and URLs from the main page
    URL = 'https://www.bbc.com/news/war-in-ukraine'
    response = requests.get(URL, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')

    articles = soup.find_all('div', class_='sc-9d830f2a-0 cKjkGu')
    headlines_and_texts = []

    for article in articles:
        # Extract headline
        headline_tag = article.find('h2', {'data-testid': 'card-headline'}) or article.find('h2') or article.find('h3')
        headline = headline_tag.get_text(strip=True) if headline_tag else "No headline"

        # Extract URL
        # Find link near headline (ancestor → in card → next)
        link_tag = None
        if headline_tag:
            link_tag = headline_tag.find_parent('a')
            if not link_tag:
                link_tag = article.find('a', href=True)
            if not link_tag:
                link_tag = headline_tag.find_next('a', href=True)

        if link_tag and link_tag.get('href'):
            url = 'https://www.bbc.com' + link_tag['href'] if link_tag['href'].startswith('/') else link_tag['href']
            print(f"URL: {url}")

            # Step 2: Fetch and parse the article page
            article_response = requests.get(url, headers=HEADERS, timeout=30)
            article_response.raise_for_status()
            article_soup = BeautifulSoup(article_response.content, 'html.parser')

            # Step 3: Extract the main text of the article
            article_body = article_soup.find('article')
            if article_body:
                paragraphs = article_body.find_all('p')
            else:
                paragraphs = []

            # Common BBC paragraph container fallback
            if not paragraphs:
                paragraphs = article_soup.select('[data-component="text-block"] p')

            full_text = "\n".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)) \
                        if paragraphs else "No article body found"

            # Save the headline and full text
            headlines_and_texts.append((headline, url, full_text))
        else:
            print(f"Skipping article with no URL: {headline}")

    if not headlines_and_texts:
        raise ValueError("No articles found. Check the source page and CSS selectors.")

    # Step 4: Save data to a text file
    with open('articles_full.txt', 'w', encoding='utf-8') as file:
        for headline, url, full_text in headlines_and_texts:
            file.write(f"Headline: {headline}\nURL:{url}\n")
            file.write(f"Full Text:\n{full_text}\n")
            file.write("-" * 80 + "\n")

#### Verify what the parser captures again

In [ ]:
# Run the function
scrape_and_save()

file_path = 'articles_full.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    lines = file.readlines()

# Print the first three sections (cells)
section_count = 0
current_section = []

print("First Three Sections from the Text File:\n")
for line in lines:
    if line.strip() == "-" * 80:  # Detect section delimiter
        section_count += 1
        print("".join(current_section))  # Print the accumulated section
        print("-" * 80)  # Re-add the delimiter for clarity
        current_section = []  # Reset for the next section
        if section_count >= 3:
            break
    else:
        current_section.append(line)  # Accumulate lines in the current section

### Step 2: Generating Visualizations

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
import re

def generate_visualizations(file_path='articles_full.txt'):
    # Step 1: Load the text file
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()

    # Step 2: Clean the text and remove URLs
    cleaned_text = re.sub(r'URL:https?\:\/\/\S+', '', text).replace('(BBC)', '')

    # Step 3: Tokenize words and filter out stop words
    stop_words = set([
        'the', 'in', 'to', 'and', 'of', 'a', 'is', 'it', 'has', 'into',
        'for', 'with', 'on', 'as', 'that', 'are', 'by', 'this', 'be',
        'from', 's', 'at', 'more', 'how', 'what', 'when', 'who', 'why',
        'was', 'were', 'but', 'an', 'their', 'them', 'after', 'before',
        'about', 'its', 'we', 'been', 'he', 'they', 'not', 'his', 'her',
        'or', 'will', 'can', 'if', 'you', 'i', 'my', 'me', 'him', 'us',
        'so', 'said', 'have', 'had', 'says', 'she', 'one', 'two', 'many',
        'full', 'there', 'there', 'all', 'text', 'now', 'out', 'which',
        'since', 'would', 'some', 're', 'headline', 'than'
    ])
    words = re.findall(r'\b\w+\b', cleaned_text.lower())
    filtered_words = [word for word in words if word not in stop_words]

    # Step 4: Perform frequency analysis on filtered words
    word_freq = Counter(filtered_words)
    top_words = word_freq.most_common(20)  # Adjust the number of words as needed
    if not top_words:
        raise ValueError("No words available to visualize. Check the scraped text.")
    words, frequencies = zip(*top_words)

    # Step 5: Generate Word Cloud
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(filtered_words))

    # Step 6: Plot Word Cloud and Bar Chart side by side
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Word Cloud
    axes[0].imshow(wordcloud, interpolation='bilinear')
    axes[0].axis('off')  # No axes for the Word Cloud
    axes[0].set_title('Word Cloud', fontsize=16)

    # Bar Chart
    bars = axes[1].bar(words, frequencies, color='skyblue')
    axes[1].set_xlabel('Word', fontsize=14)
    axes[1].set_ylabel('Frequency', fontsize=14)
    axes[1].set_title('Filtered Word Frequency', fontsize=16)
    axes[1].tick_params(axis='x', rotation=45)

    # Add annotations to the bars
    for bar in bars:
        yval = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width() / 2, yval + 1, f'{yval}', ha='center', va='bottom')

    plt.tight_layout()
    wordcloud.to_file('wordcloud_image.png')
    bar_fig, bar_ax = plt.subplots(figsize=(12, 6))
    bar_ax.bar(words, frequencies, color='skyblue')
    bar_ax.set_title('Top 20 Words')
    bar_ax.set_ylabel('Frequency')
    bar_ax.tick_params(axis='x', rotation=60)
    bar_fig.tight_layout()
    bar_fig.savefig('bar_chart.png', dpi=150, bbox_inches='tight')
    plt.close(bar_fig)
    plt.show()

#### Plot visualizations

In [ ]:
generate_visualizations()


### Step 3: Summarize articles using OpenAI GPT


In [ ]:
def generate_text_with_gpt(api_key):
    client = OpenAI(api_key=api_key)

    # Prompt embedded directly
    prompt = """
    You are a report writer summarizing news for an educational text-analysis project.
    Please summarize the texts covering the following points:
    1. Number of articles referenced,
    2. Summary of articles,
    3. Key topics covered in the texts,
    4. Frequency and the contexts of the term "UN" or "United Nations" are used,
    5. Frequency and the contexts of the term "humanitarian" are used,
    6. Humanitarian needs mentioned in the articles.
    For the section no.1, please describe it as "This section referenced ## articles published on the BBC website".
    You can use the number of articles by counting the number of URLs provided to you.
    Separate the above 6 categories into 6 separate sections by numberings with each title.
    For the section no.3, please list up in bullet point style.
    """

    with open('articles_full.txt', 'r', encoding='utf-8') as file:
        articles_text = file.read()

    # Use OpenAI's GPT to generate a summary
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": articles_text}
        ]
    )
    return completion.choices[0].message.content if completion.choices else "No summary available."

### Step 4: Compile Outputs into Word Document

In [ ]:
def compile_outputs_into_word(summary, wordcloud_path='wordcloud_image.png', bar_chart_path='bar_chart.png'):
    doc = docx.Document()
    style = doc.styles['Normal']
    font = style.font
    font.name = 'Roboto'
    font.size = Pt(11)

    today_date = datetime.datetime.now().strftime('%Y/%m/%d')
    title = "Summary of BBC articles on Ukraine " + today_date
    title_paragraph = doc.add_heading(level=0)
    title_run = title_paragraph.add_run(title)
    title_run.font.name = 'Roboto'
    title_run.font.size = Pt(24)
    title_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph("Table 1: WordCloud")
    doc.add_picture(wordcloud_path, width=Inches(5.6))
    doc.add_paragraph("Table 2: Bar Chart")
    doc.add_picture(bar_chart_path, width=Inches(5.6))
    doc.add_paragraph("Summary of articles:")
    doc.add_paragraph(summary)

    file_name = 'Article_summary.docx'
    doc.save(file_name)
    print(f"Document saved as {file_name}")

## Finally, let's save it all in a Word file.

In [ ]:
# Set OPENAI_API_KEY in your environment before launching Jupyter.
# Running this cell sends the collected text to OpenAI and may incur API charges.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set the OPENAI_API_KEY environment variable before running this cell.")
summary = generate_text_with_gpt(OPENAI_API_KEY)
compile_outputs_into_word(summary)


In [ ]:
print(summary)